In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 5.6 GPU

下载NVIDIA驱动和CUDA并设置好之后可以使用命令查看显卡信息

In [2]:
!nvidia-smi

Tue Mar 10 13:33:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 5.6.1 计算设备

In [2]:
import torch 
from torch import nn

pytorch中,CPU和GPU可用`torch.device('cpu')`和`torch.device('cuda')`表示  
应该注意的是，`cpu`设备意味着**所有**物理CPU和内存;`gpu`设备只代表**一个**卡和相应的显存.如果有多个GPU，我们使用`torch.device(f'cuda:{i}')`来表示第$i$块GPU（$i$从0开始）。  
另外，`cuda:0`和`cuda`是等价的。

In [3]:
torch.device('cpu'),torch.device('cuda'),torch.device('cuda:1')

(device(type='cpu'), device(type='cuda'), device(type='cuda', index=1))

可以查询可用的gpu数量

In [4]:
torch.cuda.device_count()

2

## 5.6.2 张量与GPU

我们可以查询张量所在设备,默认是在CPU上的

In [3]:
x=torch.tensor([1,2,3])
x.device

device(type='cpu')

需要注意的是，无论何时我们要对多个项进行操作，
它们都必须在同一个设备上。

例如，如果我们对两个张量求和，
我们需要确保两个张量都位于同一个设备上，
否则框架将不知道在哪里存储结果，甚至不知道在哪里执行计算。

### 1.存储在GPU上

有几种方法可以在GPU上存储张量。  
例如，我们可以在创建张量时指定存储设备。

在GPU上创建的张量只消耗这个GPU的显存。
我们可以使用`nvidia-smi`命令查看显存使用情况。
一般来说，我们需要确保不创建超过GPU显存限制的数据。

In [4]:
X = torch.ones(2, 3, device=torch.device('cuda:0'))
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

In [7]:
Y = torch.ones(2, 3, device=torch.device('cuda:1'))
Y

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:1')

### 2.复制

不在同一个GPU上的两个张量运算,需要先把他们放在一个GPU上. 这就需要复制  

比如前面的X和Y要相加:

In [8]:
Z=X.cuda(1) #复制
print(X)
print(Z)

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')
tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:1')


In [11]:
X+Y

RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cuda:1!

In [9]:
Y+Z

tensor([[2., 2., 2.],
        [2., 2., 2.]], device='cuda:1')

假设变量`Z`已经存在于第二个GPU上。
如果我们还是调用`Z.cuda(1)`会发生什么？
它将返回`Z`，而不会复制并分配新内存。

In [12]:
Z.cuda(1) is Z

True

### 旁注

人们使用GPU来进行机器学习，因为单个GPU相对运行速度快。
但是在设备（CPU、GPU和其他机器）之间传输数据比计算慢得多。
这也使得并行化变得更加困难，因为我们必须等待数据被发送（或者接收），
然后才能继续进行更多的操作。
这就是为什么拷贝操作要格外小心。
根据经验，多个小操作比一个大操作糟糕得多。
此外，一次执行几个操作比代码中散布的许多单个操作要好得多。
如果一个设备必须等待另一个设备才能执行其他操作，
那么这样的操作可能会阻塞。
这有点像排队订购咖啡，而不像通过电话预先订购：
当客人到店的时候，咖啡已经准备好了。

最后，当我们打印张量或将张量转换为NumPy格式时，
如果数据不在内存中，框架会首先将其复制到内存中，
这会导致额外的传输开销。
更糟糕的是，它现在受制于全局解释器锁，使得一切都得等待Python完成。

## 5.6.3 神经网络与GPU

类似地，神经网络模型可以指定设备。
下面的代码将模型参数放在GPU上。

In [5]:
net = nn.Sequential(nn.Linear(3, 1))
net = net.to(device='cuda:0')

当输入为GPU上的张量时，模型将在同一GPU上计算结果。

In [6]:
net(X)

tensor([[-0.2399],
        [-0.2399]], device='cuda:0', grad_fn=<AddmmBackward0>)

让我们**确认模型参数存储在同一个GPU上**

In [7]:
net[0].weight.data.device

device(type='cuda', index=0)

## 小结

* 我们可以指定用于存储和计算的设备，例如CPU或GPU。默认情况下，数据在主内存中创建，然后使用CPU进行计算。
* 深度学习框架要求计算的所有输入数据都在同一设备上，无论是CPU还是GPU。
* 不经意地移动数据可能会显著降低性能。一个典型的错误如下：计算GPU上每个小批量的损失，并在命令行中将其报告给用户（或将其记录在NumPy `ndarray`中）时，将触发全局解释器锁，从而使所有GPU阻塞。最好是为GPU内部的日志分配内存，并且只移动较大的日志。

## 练习

1. 尝试一个计算量更大的任务，比如大矩阵的乘法，看看CPU和GPU之间的速度差异。再试一个计算量很小的任务呢？
1. 我们应该如何在GPU上读写模型参数？
1. 测量计算1000个$100 \times 100$矩阵的矩阵乘法所需的时间，并记录输出矩阵的Frobenius范数，一次记录一个结果，而不是在GPU上保存日志并仅传输最终结果。
1. 测量同时在两个GPU上执行两个矩阵乘法与在一个GPU上按顺序执行两个矩阵乘法所需的时间。提示：应该看到近乎线性的缩放。


----

### 1. 尝试一个计算量更大的任务，比如大矩阵的乘法，看看CPU和GPU之间的速度差异。再试一个计算量很小的任务呢？

In [8]:
import torch
import time

# 检查 GPU 是否可用
device_cpu = torch.device('cpu')
device_gpu = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device_gpu} (GPU available: {torch.cuda.is_available()})")

# 定义矩阵大小
size_large = 5000   # 大矩阵：5000x5000
size_small = 2      # 小矩阵：2x2

Using device: cuda (GPU available: True)


In [10]:
print("\n=== 大矩阵乘法 (5000x5000) ===")

# 在 CPU 上生成随机矩阵
A_large_cpu = torch.randn(size_large, size_large, device=device_cpu)
B_large_cpu = torch.randn(size_large, size_large, device=device_cpu)

# CPU 计时
start = time.time()
C_large_cpu = torch.mm(A_large_cpu, B_large_cpu)
end = time.time()
print(f"CPU time: {end - start:.4f} seconds")

if torch.cuda.is_available():
    # 将矩阵移到 GPU
    A_large_gpu = A_large_cpu.to(device_gpu)
    B_large_gpu = B_large_cpu.to(device_gpu)
    
    # 预热 GPU（可选）
    torch.mm(A_large_gpu, B_large_gpu)
    torch.cuda.synchronize()

    # GPU 计时（包含数据传输时间）
    start = time.time()
    A_large_gpu = A_large_cpu.to(device_gpu)
    B_large_gpu = B_large_cpu.to(device_gpu)
    C_large_gpu = torch.mm(A_large_gpu, B_large_gpu)
    torch.cuda.synchronize()
    end = time.time()
    print(f"GPU time (with data transfer): {end - start:.4f} seconds")

    # 仅计算时间（数据已在 GPU）
    A_large_gpu = A_large_cpu.to(device_gpu)
    B_large_gpu = B_large_cpu.to(device_gpu)
    torch.cuda.synchronize()
    start = time.time()
    C_large_gpu = torch.mm(A_large_gpu, B_large_gpu)
    torch.cuda.synchronize()
    end = time.time()
    print(f"GPU time (compute only): {end - start:.4f} seconds")
else:
    print("GPU not available, skipping GPU tests.")


=== 大矩阵乘法 (5000x5000) ===
CPU time: 1.0005 seconds
GPU time (with data transfer): 0.1294 seconds
GPU time (compute only): 0.0534 seconds


### 2. 我们应该如何在GPU上读写模型参数？
使用`.to(device)`将模型移动到GPU上
```python
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
net=nn.Linear(4,2)
net.to(device)
```

### 3. 测量计算1000个$100 \times 100$矩阵的矩阵乘法所需的时间，并记录输出矩阵的Frobenius范数
比较一次记录一个结果，和在GPU上保存日志并仅传输最终结果。

In [24]:
import torch
import time

# 检查GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cpu':
    raise RuntimeError("需要GPU来演示")

# 参数
N = 1000          # 矩阵乘法次数
size = 100        # 矩阵维度

A = torch.randn(size, size, device=device)
B = torch.randn(size, size, device=device)

def method_inefficient():
    """低效方法：每次计算后立即将范数传回CPU（触发同步）"""
    norms = []                     # Python列表，位于CPU
    for i in range(N):
        C = torch.matmul(A, B)
        norm = C.norm().item()     # .item() 将GPU标量复制到CPU并阻塞直到计算完成
        norms.append(norm)
    return norms

def method_efficient():
    """高效方法：在GPU上累积所有范数，最后一次性传回CPU"""
    norms_gpu = torch.zeros(N, device=device)   # 预先分配GPU张量
    for i in range(N):
        C = torch.matmul(A, B)
        norms_gpu[i] = C.norm()                  # 仅在GPU上赋值，无CPU传输
    norms = norms_gpu.cpu().numpy()              # 最后一次性传回CPU
    return norms

# 使用CUDA事件精确计时
def time_method(method, name):
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)
    
    torch.cuda.synchronize()
    start_event.record()
    result = method()
    end_event.record()
    torch.cuda.synchronize()
    
    elapsed = start_event.elapsed_time(end_event) / 1000.0  # 转换为秒
    print(f"{name} 耗时: {elapsed:.4f} 秒")
    return result

# 运行对比
print("运行低效方法...")
res_ineff = time_method(method_inefficient, "低效方法")

print("运行高效方法...")
res_eff = time_method(method_efficient, "高效方法")

# 验证结果一致（可选）
print(f"前5个范数（低效）: {res_ineff[:5]}")
print(f"前5个范数（高效）: {res_eff[:5]}")

运行低效方法...
低效方法 耗时: 0.0626 秒
运行高效方法...
高效方法 耗时: 0.0520 秒
前5个范数（低效）: [999.8231201171875, 999.8231201171875, 999.8231201171875, 999.8231201171875, 999.8231201171875]
前5个范数（高效）: [999.8231 999.8231 999.8231 999.8231 999.8231]


### 4. 测量同时在两个GPU上执行两个矩阵乘法与在一个GPU上按顺序执行两个矩阵乘法所需的时间。
提示：应该看到近乎线性的缩放。

In [20]:
import torch

def time_matmul(device1, device2=None, parallel=True):
    """测量矩阵乘法时间。若parallel=True且device2不为None，则在两个设备上并行执行两个乘法；
       否则在device1上顺序执行两个乘法。"""
    size = 2000  # 矩阵大小，确保计算时间显著
    A1 = torch.randn(size, size, device=device1)
    B1 = torch.randn(size, size, device=device1)
    A2 = torch.randn(size, size, device=device2 if device2 else device1)
    B2 = torch.randn(size, size, device=device2 if device2 else device1)

    # 创建事件用于计时
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)
    
    if parallel and device2 is not None:
        # 并行执行：两个设备上的计算同时开始
        start.record()
        # 在默认流中启动第一个GPU上的乘法
        C1 = torch.matmul(A1, B1)
        # 在第二个GPU上使用单独的流（或直接调用，但需确保并发）
        with torch.cuda.device(device2):
            # 可选的流，但默认流在不同设备上会自动并发
            C2 = torch.matmul(A2, B2)
        end.record()
    else:
        # 顺序执行：在第一个GPU上依次做两次乘法
        start.record()
        C1 = torch.matmul(A1, B1)
        C2 = torch.matmul(A2, B2)  # 仍在device1上
        end.record()
    
    torch.cuda.synchronize(device1)
    if device2:
        torch.cuda.synchronize(device2)
    elapsed = start.elapsed_time(end) / 1000.0  # 秒
    return elapsed

if torch.cuda.device_count() < 2:
    print("需要至少两个GPU来演示并行加速")
else:
    device0 = torch.device('cuda:0')
    device1 = torch.device('cuda:1')
    
    # 并行时间
    t_parallel = time_matmul(device0, device1, parallel=True)
    # 顺序时间（在同一GPU上）
    t_sequential = time_matmul(device0, parallel=False)
    
    print(f"顺序执行时间: {t_sequential:.4f} 秒")
    print(f"并行执行时间: {t_parallel:.4f} 秒")
    print(f"加速比: {t_sequential / t_parallel:.2f}x (接近2x为线性缩放)")

顺序执行时间: 0.0117 秒
并行执行时间: 0.0060 秒
加速比: 1.96x (接近2x为线性缩放)
